# Words prediction

In [ ]:
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Sample text data
text = """
LSTMs are a type of recurrent neural network capable of learning long-term dependencies.
They work well on sequence prediction problems such as text generation.
"""

# Parameters
sequence_length = 5  # Number of words in each input sequence

# Step 1: Tokenize the text
tokenizer = Tokenizer()
tokenizer.fit_on_texts([text])
word_index = tokenizer.word_index
vocab_size = len(word_index) + 1  # Adding 1 for padding

# Convert text to sequences of integers
encoded = tokenizer.texts_to_sequences([text])[0]

# Step 2: Create input-output pairs
sequences = []
for i in range(sequence_length, len(encoded)):
    seq = encoded[i - sequence_length:i]  # Input sequence
    target = encoded[i]  # Next word
    sequences.append(seq + [target])  # Append as a single list (input + target)

# Convert to numpy array
sequences = np.array(sequences)

# Separate inputs and outputs
X, y = sequences[:, :-1], sequences[:, -1]

# Step 3: Pad sequences (if needed)
X = pad_sequences(X, maxlen=sequence_length - 1, padding='pre')

# Convert outputs to categorical
from tensorflow.keras.utils import to_categorical
y = to_categorical(y, num_classes=vocab_size)

# Dataset prepared
print(f"Vocabulary size: {vocab_size}")
print(f"Input shape: {X.shape}")
print(f"Output shape: {y.shape}")

# Example: Create an LSTM model
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=50, input_length=sequence_length - 1),
    LSTM(100),
    Dense(vocab_size, activation='softmax')
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

# Example training
model.fit(X, y, epochs=20, batch_size=32)


Vocabulary size: 25
Input shape: (20, 4)
Output shape: (20, 25)


/usr/local/lib/python3.10/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding (Embedding)                │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm (LSTM)                          │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ ?                           │     0 (unbuilt) │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step - accuracy: 0.0000e+00 - loss: 3.2200
Epoch 2/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 0.0500 - loss: 3.2159
Epoch 3/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step - accuracy: 0.1000 - loss: 3.2117
Epoch 4/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step - accuracy: 0.2000 - loss: 3.2076
Epoch 5/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step - accuracy: 0.2000 - loss: 3.2034
Epoch 6/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step - accuracy: 0.3000 - loss: 3.1990
Epoch 7/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step - accuracy: 0.3500 - loss: 3.1946
Epoch 8/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - accuracy: 0.5500 - loss: 3.1900
Epoch 9/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step - accuracy: 0.6500 - loss: 3.1852
Epoch 10/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 60ms/step - accuracy: 0.6500 - loss: 3.1802
Epoch 11/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - accuracy: 0.6500 - loss: 3.1749
Epoch 12/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step - accuracy: 0.7000 - loss: 3.1693

In [ ]:
import numpy as np
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Function to map integers back to words
reverse_word_index = {value: key for key, value in tokenizer.word_index.items()}

def generate_text(model, tokenizer, seed_text, next_words, sequence_length):
    for _ in range(next_words):
        # Tokenize the seed text
        tokenized = tokenizer.texts_to_sequences([seed_text])[0]

        # Pad the sequence to match the input length of the model
        tokenized = pad_sequences([tokenized], maxlen=sequence_length - 1, padding='pre')

        # Predict the next word
        predicted_probabilities = model.predict(tokenized, verbose=0)
        predicted_word_index = np.argmax(predicted_probabilities, axis=-1)[0]

        # Map predicted index back to word
        predicted_word = reverse_word_index.get(predicted_word_index, "")

        # Append the predicted word to the seed text
        seed_text += " " + predicted_word

    return seed_text

# Example usage
seed_text = "LSTMs are a type"
next_words_to_predict = 10  # Predict the next 5 words

generated_text = generate_text(model, tokenizer, seed_text, next_words_to_predict, sequence_length)
print(f"Generated text: {generated_text}")


Generated text: LSTMs are a type recurrent neural neural network of learning learning long term they
